In [ ]:
import re
import string

letters_upper = [
    "A", "B", "C", "Ç", "D", "E", "Ə", "F", "G", "Ğ",
    "H", "X", "I", "İ", "J", "K", "Q", "L", "M", "N",
    "O", "Ö", "P", "R", "S", "Ş", "T", "U", "Ü", "V",
    "Y", "Z"
]

abbreviations = {
    "dr", "prof", "c", "səh", "cə", "s", "m", "km", "kg",
    "nr", "tel", "vs", "həm", "b", "bb", "bax", "bk",
}

def extract_features(left_context, right_context):
    # ensure inputs are strings
    left_context = str(left_context)
    right_context = str(right_context)

    # last word before dot
    left_words = re.findall(r'\b\w+\b', left_context)
    prev_word = left_words[-1] if left_words else ""

    # first word after dot
    right_words = re.findall(r'\b\w+\b', right_context)
    next_word = right_words[0] if right_words else ""

    features = {}

    # 1. previous word length 1
    features["prev_len_1"] = int(len(prev_word) == 1)

    # 2. next word length 1
    features["next_len_1"] = int(len(next_word) == 1)

    # 3. previous word capitalized
    features["prev_capitalized"] = int(
        len(prev_word) > 0 and prev_word[0] in letters_upper
    )

    # 4. next word capitalized
    features["next_capitalized"] = int(
        len(next_word) > 0 and next_word[0] in letters_upper
    )

    # 5. previous word abbreviation
    features["prev_abbrev"] = int(prev_word.lower() in abbreviations)

    # 6. next word abbreviation
    features["next_abbrev"] = int(next_word.lower() in abbreviations)

    # 7. number before dot
    features["number_before_dot"] = int(prev_word.isdigit())

    # 8. number after dot
    features["number_after_dot"] = int(next_word.isdigit())

    # 9. character immediately before dot
    prev_char = left_context[-1] if left_context else ""
    features["punct_before_dot"] = int(prev_char in string.punctuation)

    # 10. first non-space character after dot
    stripped = right_context.lstrip()
    next_char = stripped[0] if stripped else ""
    features["punct_after_dot"] = int(next_char in string.punctuation)

    return features

In [10]:
import numpy as np
from numpy import linalg as LA

def sigmoid(x):
    return 1/(1 + np.exp(-x))

def gradient_descend(X_batch, y_pred, y_result, weights, biases, eta, batch_size, l1_lambda = 0.0, l2_lambda = 0.0):
    grad_w = (1/batch_size) * X_batch.T @ (y_pred - y_result) 
    grad_b = (1/batch_size) * np.sum(y_pred - y_result)

    if l2_lambda > 0 and l1_lambda == 0.0:
        grad_w += l2_lambda * weights
    
    if l1_lambda > 0 and l2_lambda == 0.0:
        grad_w += l1_lambda * np.sign(weights)
    weights -= eta * grad_w
    biases -= eta * grad_b
    return weights, biases

In [11]:
batch_size = 10
feature_length = 10
rng = np.random.default_rng(42)

weights = rng.normal(0, 0.01, feature_length)
bias = 0.0

In [ ]:
import pandas as pd

eta = 0.01
data = pd.read_csv("sentence_boundary_data.csv")

current_line = 0

while current_line != data.shape[0]:
    text_batch = data.iloc[current_line : current_line + batch_size]

    current_line += min(current_line + batch_size, data.shape[0] - batch_size)

    if current_line - data.shape[0] > 0: break

    X_batch = []
    y_result = []

    for line in text_batch.itertuples(index=False):
        X_batch.append(list(extract_features(line[0], line[1]).values()))
        y_result.append(line[2])

    X_batch = np.array(X_batch)
    y_result = np.array(y_result)

    y_pred = sigmoid(X_batch @ weights + bias)

    weights, bias = gradient_descend(X_batch, y_pred, y_result, weights, bias, eta, batch_size, 0, 2)

print(weights, bias)


IndexError: list index out of range

In [ ]:
with open("/Users/elmarmammadov/Documents/NLP/CleanTextOCR/Cafa Cabbarli/Eserleri_I_cild.md", 'r') as file:
    test = file.read()

import string

tokens = re.findall(r'\w+|[^\w\s]', test)
window = 1
sentences = []
current_sentence = []
results = []

for i, token in enumerate(tokens):
    current_sentence.append(token)  # add token to current sentence
    
    if token == ".":
        # compute features
        left_context = tokens[max(0, i-window):i]
        right_context = tokens[i+1:i+1+window]

        features = list(extract_features(left_context, right_context).values())
        p = sigmoid(np.dot(weights, features) + bias)

        if p > 0.5:
            # join current_sentence tokens into a string and save
            sentences.append(" ".join(current_sentence).replace(" .", "."))
            current_sentence = []  # reset for next sentence

        results.append({
            "left": left_context,
            "right": right_context,
            "label": 1 if p > 0.5 else 0
        })



with open("result_segmentation2.txt", "w") as file:
    for sentence in sentences:
        file.write(sentence + "\n")

In [ ]:
text = input("enter text")
tokens = re.findall(r'\w+|[^\w\s]', text)
window = 1
sentences2 = []
current_sentence = []
results = []

for i, token in enumerate(tokens):
    current_sentence.append(token)  # add token to current sentence
    
    if token == ".":
        # compute features
        left_context = tokens[max(0, i-window):i]
        right_context = tokens[i+1:i+1+window]

        features = list(extract_features(left_context, right_context).values())
        p = sigmoid(np.dot(weights, features) + bias)

        if p > 0.5:
            # join current_sentence tokens into a string and save
            sentences2.append(" ".join(current_sentence).replace(" .", "."))
            current_sentence = []  # reset for next sentence

        results.append({
            "left": left_context,
            "right": right_context,
            "label": 1 if p > 0.5 else 0
        })

print(sentences2)

['Məktəbli Cəfərin uşaqlığı , yaradıcılığa başladığı ilk illər haqqında onun sevimli müəll mi Abdulla Şaiqin xatirələri də maraqlıdır : “ Cəfər Cabbarlını hələ kiçik ikən , Bakının yeddinci şəhər məktəbində oxuduğu zamandan tanıyırdım.', 'O mənim ən çox sevdiyim şagirdlərimdən idi.', 'O , ilk şerini çox zaman hamıdan əvvəl mənə oxuyar , mənim göstərişlərim əsasında işləyərdi.', 'Onda möhkəm iradə və nikbinlik vardı.', 'Heç bir müvəffəqiyyətsizlik onu ruhdan salmaz , əksinə , şeirlərinin üzərində daha səylə çalışmasına səbəb olardı.', 'Məsləhət və göstərişləri o , çox asanlıqla qavrayar və əməl edərdi.', 'Ehtimal ki , Cəfərin ilk şeirlərinin əksəriyyəti çap olunmamışdır.', '” ( Abdulla Şaiq.', 'Xatirələrim , Bakı , “ Gənclik ” , 1973 , səh.', '286 - 287 ).']
